In [ ]:
import os, json, time, shutil, hashlib, gc
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from IPython.display import display
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
TASK = Path.cwd().parent
PROFILE = os.environ.get('NEURAL_DNA_PROFILE', 'local')
INPUT, OUTPUT = TASK / 'input' / PROFILE, TASK / 'output' / PROFILE
OUTPUT.mkdir(parents=True, exist_ok=True)

In [ ]:
cfg = json.loads((INPUT / 'config.json').read_text())
SEED = cfg['seed']
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(cfg['threads'])

In [ ]:
for cell in json.loads((INPUT / 'helpers.ipynb').read_text())['cells']:
    exec(''.join(cell['source']), globals())
carry(INPUT, OUTPUT)

In [ ]:
summary = pd.read_csv(INPUT / 'summary.csv')
queries = pd.read_csv(INPUT / 'queries.csv')
index = pd.read_csv(INPUT / 'prediction_index.csv')
truth = np.load(INPUT / 'truth.npz')['ancestry']
with np.load(INPUT / 'predictions.npz') as data:
    predictions = {key: data[key] for key in data.files}

In [ ]:
sns.set_theme(style='whitegrid', palette='colorblind')
order = ['global', 'nearest', 'copying']
palette = dict(zip(order, sns.color_palette('colorblind', 3)))
last_step = max(cfg['finetune_steps'])
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for ax, kind in zip(axes, ['long', 'short']):
    selected = summary.query("panel == 'proxy' and steps == @last_step and kind == @kind")
    sns.lineplot(data=selected, x='markers', y='block_accuracy', hue='method', style='method',
        hue_order=order, style_order=order, palette=palette, markers=True, dashes=True, ax=ax)
    ax.set(xscale='log', ylim=(0, 1.03), title=f'{kind.capitalize()} source runs',
        xlabel='Sampled parameters per block', ylabel='Historical source accuracy')
axes[0].get_legend().remove()
fig.suptitle(f'{PROFILE}: related references, after {last_step} training steps')
fig.tight_layout()
fig.savefig(OUTPUT / 'source_recovery.png', dpi=180, bbox_inches='tight')
plt.show()

In [ ]:
# fixed first test example, before and after training
fig, axes = plt.subplots(2, 1, figsize=(11, 4))
for ax, steps in zip(axes, [cfg['finetune_steps'][0], cfg['finetune_steps'][-1]]):
    selected = index.query("mosaic_id == 'test_long_000' and panel == 'proxy' and markers == 4 and steps == @steps")
    q = int(selected.query_id.iloc[0])
    calls = [truth[q]] + [predictions['labels'][int(selected.query('method == @method').prediction_id.iloc[0])] for method in ['nearest', 'copying']]
    paint = pd.DataFrame(calls, index=['True', 'Nearest', 'Copying'], columns=np.arange(1, len(truth[q]) + 1))
    sns.heatmap(paint, cmap=sns.color_palette('colorblind', 2), vmin=0, vmax=1,
        annot=np.array(['A', 'B'])[paint.to_numpy()], fmt='', cbar=False, ax=ax)
    ax.set(title=f'{steps} further training steps', xlabel='Transformer block', ylabel='')
    ax.tick_params(axis='y', rotation=0)
fig.tight_layout()
fig.savefig(OUTPUT / 'ancestry_painting.png', dpi=180, bbox_inches='tight')
plt.show()

In [ ]:
def markdown_table(frame):
    lines = ['| ' + ' | '.join(frame.columns) + ' |', '| ' + ' | '.join(['---'] * len(frame.columns)) + ' |']
    lines += ['| ' + ' | '.join(map(str, row)) + ' |' for row in frame.itertuples(index=False, name=None)]
    return '\n'.join(lines)

headline = summary.query('markers == 4')[['panel', 'steps', 'kind', 'method', 'n', 'block_accuracy', 'boundary_f1', 'perplexity']].copy()
for column in ['block_accuracy', 'boundary_f1', 'perplexity']:
    headline[column] = headline[column].map(lambda x: f'{x:.4f}')
info = json.loads((INPUT / 'training_info.json').read_text())
fixture_note = 'This is an offline random-model integration fixture, not a pretrained language-model result.' if cfg['backend'] == 'random_fixture' else 'This is a small pretrained causal-language-model pilot.'
report = f'''# Language-model ancestry: {PROFILE}

{fixture_note}

Model: {cfg['model_id'] if cfg['backend'] == 'pretrained' else 'random Llama fixture'}.
Seed: 63. Transformer blocks: {truth.shape[1]}. Model parameters: {info['model_parameters']:,}.

The experiment trains two source lineages and independent reference/calibration/test children, copies known block runs, and continues training the mosaics. All transformer-block parameters are trainable; embeddings, tied output head, and final norm are frozen. Source checkpoints store FP16 updates and are reconstructed before exact fingerprints are recorded.

Calibration and test donors are different branches. Settings are fitted on calibration ancestry only. The inference notebook never reads test truth. The language-model evaluation uses held-out corpus windows, reused across checkpoints.

## Four sampled parameters per block

{markdown_table(headline)}

All marker counts and per-mosaic differences are in `summary.csv` and `paired_gains.csv`. Nearest, global, and copying comparisons share the same observations. The saved two-source convex least-squares mixture coefficients are a diagnostic: with exact reference endpoints, their hard source decisions equal nearest matching mathematically; they are not an independent competing classifier.

These small sample counts and one source panel support a local pilot only. A benefit on long runs may come with errors on short runs. Copied source labels describe historical block origin, not causal influence on generated tokens. Marker drift is measured on sampled coordinates; perplexity uses the saved held-out windows. No hypothesis-test or population-level generalization claim is made.

The local pilot does not include absent-source rejection, invariance to hidden-coordinate changes, multiple model families, or the full modelDNA implementation. The copying objective is a standard mismatch-plus-switching model, so the genetics interpretation alone does not establish methodological novelty.
'''
(OUTPUT / 'RESULTS.md').write_text(report)
display(pd.DataFrame({'profile': [PROFILE], 'backend': [cfg['backend']], 'report': [str(OUTPUT / 'RESULTS.md')]}))